Les algorithmes du Machine Learning  et le Fight des IA


Phase 0: La mise en route :

Phase B : Prédire les prix immobiliers (régression)

In [4]:
import numpy as np
import pandas as pd

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# Les Prix de l'immobilier (en régression)

def charger_immobilier():
    data = fetch_california_housing()
    X, y = data.data, data.target
    print(f"Shape : {X.shape} | Variables : {data.feature_names}")
    print("Cible : prix médian en centaines de milliers de $")
    return X, y

def evaluer_regression(nom, modele, X_train, X_test, y_train, y_test):
    modele.fit(X_train, y_train)
    y_pred = modele.predict(X_test)
    r2   = r2_score(y_test, y_pred)
    mae  = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    print(f"{nom:<20} R2={r2:.2f}  MAE={mae:.2f}  RMSE={rmse:.2f}")
    return {"r2": r2, "mae": mae, "rmse": rmse}

# execution du codes
X, y = charger_immobilier()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s  = scaler.transform(X_test)

#rappelle le random state = 42 est une valeur par "defaut", cela permet par exemple à un groupe en fixant 42 de débuter toujours avec un split identique 
# Nous pourrions tous débuté à 26 

evaluer_regression("La régression linéaire",  LinearRegression(),                         X_train_s, X_test_s, y_train, y_test)
evaluer_regression("Le random forest",      RandomForestRegressor(n_estimators=200, random_state=42), X_train_s, X_test_s, y_train, y_test)

# ── Checkpoints qualité Phase A ───────────────────────────────────────────────

# --- Cas limite : 100 lignes ---
X_train_100, X_test_100, y_train_100, y_test_100 = train_test_split(
    X[:100], y[:100], test_size=0.2, random_state=42
)
scaler_100 = StandardScaler().fit(X_train_100)
evaluer_regression("LinReg 100 lignes",
                   LinearRegression(),
                   scaler_100.transform(X_train_100),
                   scaler_100.transform(X_test_100),
                   y_train_100, y_test_100)

# le R2 s'effondre completement, on tombe a des valeurs vraiment mauvaises
# voire négatives parfois. c'est logique en fait : le modele a vu 80 exemples
# pour apprendre les prix de TOUTE la californie, c'est beaucoup trop peu.
# il n'a pas eu assez de diversité dans les données pour capturer les patterns
# (quartiers riches, zones rurales, bord de mer etc...)
# => un modele a besoin de données pour generaiser, 80 lignes c'est rien


# --- Cas adversarial : quartier fictif hors plage ---
# ordre des variables : MedInc, HouseAge, AveRooms, AveBedrms,
#                       Population, AveOccup, Latitude, Longitude
quartier_fictif = np.array([[0, 20, 5, 1, 9000, 3, 36.5, -119.0]])
quartier_scaled = scaler.transform(quartier_fictif)

# on réutilise le modele déja entrainé sur le dataset complet
pred_lr = LinearRegression().fit(
    scaler.transform(X_train), y_train).predict(quartier_scaled)[0]
print(f"Prédiction LR  : {pred_lr:.2f} (x100k$)")

# la valeur sortie est completement absurde, le modele s'en fout que
# le revenu median soit a 0 ou que la population soit de 9000 personnes,
# il extrapole juste dans le vide sans aucun garde fou.
# en prod ca serait un vrai probleme : on pourrait se retrouver a afficher
# un prix negatif ou un prix de 50 millions a un utilisateur
# ce qu'on devrait faire :
#   - verifier que les valeurs d'entrée sont dans une plage "normale" (celle du train)
#   - rejeter la requete si c'est hors plage et renvoyer une erreur clair
#   - jamais faire confiance a ce que l'utilisateur envoie sans validation

Shape : (20640, 8) | Variables : ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']
Cible : prix médian en centaines de milliers de $
La régression linéaire R2=0.58  MAE=0.53  RMSE=0.75
Le random forest     R2=0.81  MAE=0.33  RMSE=0.50
LinReg 100 lignes    R2=0.71  MAE=0.38  RMSE=0.49
Prédiction LR  : -0.40 (x100k$)


Phase B: 